## Prepare data

We first download the data. Here, I'm just using the IAM test set, as this was released by the TrOCR authors in the unilm repository. It can be downloaded from [this page](https://github.com/microsoft/unilm/tree/master/trocr). 

Let's make a [regular PyTorch dataset](https://pytorch.org/tutorials/beginner/basics/data_tutorial.html). We first create a Pandas dataframe with 2 columns. Each row consists of the file name of an image, and the corresponding text.

In [ ]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel, AutoTokenizer, TrOCRForCausalLM
from peft import LoraConfig, get_peft_model, TaskType

# For example, using a TrOCR model pre-trained on handwritten text:
processor = TrOCRProcessor.from_pretrained("Ransaka/sinhala-ocr-model-v3", use_fast=True)
tokenizer = AutoTokenizer.from_pretrained("Ransaka/sinhala-ocr-model-v3")
processor.tokenizer = tokenizer

# Load the pre-trained TrOCR model:
model = VisionEncoderDecoderModel.from_pretrained("Ransaka/sinhala-ocr-model-v3")

preprocessor_config.json:   0%|          | 0.00/325 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/315 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/273 [00:00<?, ?B/s]

In [316]:
# --- PEFT/LoRA integration ---
# Configure LoRA for encoder and decoder separately
dora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    lora_dropout=0.01,
    use_dora=True,
    target_modules=["query", "value"], # or try ["key", "query", "value"]
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    lora_dropout=0.01,
    bias="none",
    use_dora=True,
    target_modules=["q_proj", "v_proj"]  # Adjust these names if needed (check your model's module names)
)

# Apply LoRA to encoder and decoder
model.encoder = get_peft_model(model.encoder, dora_config)
model.decoder = get_peft_model(model.decoder, lora_config)

In [317]:
# Set special tokens and config
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size
model.config.eos_token_id = processor.tokenizer.sep_token_id
model.config.max_length = 32
model.config.early_stopping = True
model.config.no_repeat_ngram_size = 3
model.config.length_penalty = 0.8
model.config.num_beams = 2
model.config.image_size = 128

In [318]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params: {trainable_params} / Total params: {total_params}")

Trainable params: 1124352 / Total params: 316342784


In [275]:
from peft import PeftModel
import os

def save_peft_model(model, output_dir, epoch=None):
    """
    Save a PEFT model's adapter weights properly to allow continued training.
    
    Args:
        model: The PEFT-adapted model
        output_dir: Directory to save the model
        epoch: Optional epoch number to include in path
    """
    if epoch is not None:
        save_dir = os.path.join(output_dir, f"epoch_{epoch}")
    else:
        save_dir = output_dir
    
    # Create directory if it doesn't exist
    os.makedirs(save_dir, exist_ok=True)
    
    # Save encoder adapters
    encoder_dir = os.path.join(save_dir, "encoder_adapters")
    os.makedirs(encoder_dir, exist_ok=True)
    model.encoder.save_pretrained(encoder_dir)
    
    # Save decoder adapters
    decoder_dir = os.path.join(save_dir, "decoder_adapters")
    os.makedirs(decoder_dir, exist_ok=True)
    model.decoder.save_pretrained(decoder_dir)
    
    # Also save the base config
    model.config.save_pretrained(save_dir)
    
    print(f"PEFT model adapters saved to {save_dir}")
    return save_dir

def load_peft_model(base_model_path, adapter_path):
    """
    Load a PEFT model with adapters for continued training.
    
    Args:
        base_model_path: Path to the base pre-trained model
        adapter_path: Path to the directory containing adapter weights
    
    Returns:
        The PEFT model with adapters loaded
    """
    # Load base model
    model = VisionEncoderDecoderModel.from_pretrained(base_model_path)
    
    # Configure the encoder and decoder with PEFT
    dora_config = LoraConfig(
        r=8,
        lora_alpha=32,
        lora_dropout=0.01,
        use_dora=True,
        target_modules=["query", "value"],
    )

    lora_config = LoraConfig(
        r=8,
        lora_alpha=32,
        lora_dropout=0.01,
        bias="none",
        use_dora=True,
        target_modules=["q_proj", "v_proj"]
    )
    
    # Apply PEFT to encoder and decoder
    model.encoder = get_peft_model(model.encoder, dora_config)
    model.decoder = get_peft_model(model.decoder, lora_config)
    
    # Load adapter weights if they exist
    encoder_adapter_path = os.path.join(adapter_path, "encoder_adapters")
    decoder_adapter_path = os.path.join(adapter_path, "decoder_adapters")
    
    if os.path.exists(encoder_adapter_path):
        print(f"Loading encoder adapters from {encoder_adapter_path}")
        model.encoder.load_adapter(encoder_adapter_path)
    
    if os.path.exists(decoder_adapter_path):
        print(f"Loading decoder adapters from {decoder_adapter_path}")
        model.decoder.load_adapter(decoder_adapter_path)
    
    return model

In [42]:
from torch.optim import AdamW
from tqdm.notebook import tqdm

# Load a partially trained model or start fresh
resume_training = False  # Set to True to resume from a checkpoint
checkpoint_path = "/workspace/pandith/model/checkpoint"

if resume_training and os.path.exists(checkpoint_path):
    print(f"Resuming training from checkpoint: {checkpoint_path}")
    # Load model with adapters
    model = load_peft_model("ds1219/Pandithaya-Printed-Words-128", checkpoint_path)
    model.compile()
    # Configure model settings
    model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
    model.config.pad_token_id = processor.tokenizer.pad_token_id
    model.config.vocab_size = model.config.decoder.vocab_size
    model.config.eos_token_id = processor.tokenizer.sep_token_id
    model.config.max_length = 32
    model.config.early_stopping = True
    model.config.no_repeat_ngram_size = 3
    model.config.length_penalty = 0.8
    model.config.num_beams = 2
    model.config.image_size = 128

optimizer = AdamW(model.parameters(), lr=5e-5)

# Create adaptive learning rate scheduler that reduces LR when validation metric stops improving
scheduler = lr_scheduler.ReduceLROnPlateau(
    optimizer, 
    mode='min',              # Lower CER is better
    factor=0.5,              # Reduce LR by half when plateauing
    patience=2,              # Wait for 2 epochs before reducing LR
    threshold=0.01,          # Significant change threshold
    threshold_mode='rel',    # Relative threshold
    min_lr=1e-6              # Don't go below this LR
)

best_cer = float('inf')
for epoch in range(10):  # loop over the dataset multiple times
   model.to(device)
   model.train()
   train_loss = 0.0
   for batch in tqdm(train_dataloader):
      for k,v in batch.items():
        batch[k] = v.to(device)
      outputs = model(**batch,interpolate_pos_encoding=True)
      loss = outputs.loss
      loss.backward()
      optimizer.step()
      optimizer.zero_grad()
      train_loss += loss.item()

   avg_train_loss = train_loss/len(train_dataloader)
   print(f"Loss after epoch {epoch}:", avg_train_loss)
   print(f"Current learning rate: {get_current_lr(optimizer)}")
   
   # Save checkpoint after each epoch for resuming training later
   save_peft_model(model, "/workspace/pandith/model/checkpoint", epoch)
   
   # Create a merged model copy for evaluation
   model2 = deepcopy(model)
   model2.encoder = model2.encoder.merge_and_unload()
   model2.decoder = model2.decoder.merge_and_unload()
   model2.to(device)

   # evaluate
   model.eval()
   valid_cer = 0.0
   with torch.no_grad():
     for batch in tqdm(eval_dataloader):
       for k,v in batch.items():
         batch[k] = v.to(device)
       outputs = model2.generate(batch["pixel_values"],interpolate_pos_encoding=True)
       cer = compute_cer(pred_ids=outputs, label_ids=batch["labels"])
       valid_cer += cer 
   
   avg_valid_cer = valid_cer / len(eval_dataloader)
   print("Validation CER:", avg_valid_cer)

   del(model2)
   torch.cuda.empty_cache()
   
   # Store current LR to detect changes
   old_lr = get_current_lr(optimizer)
   
   # Step the scheduler with validation metric
   scheduler.step(avg_valid_cer)
   
   # Check if LR changed and print message
   new_lr = get_current_lr(optimizer)
   if new_lr != old_lr:
       print(f"Learning rate reduced from {old_lr} to {new_lr}")
   
   # Save model if it's the best so far - save both merged model and PEFT adapters
   if avg_valid_cer < best_cer:
     best_cer = avg_valid_cer
     print(f"New best CER: {best_cer}! Saving model...")
     
     # Save PEFT adapters for continued training
     save_peft_model(model, "/workspace/pandith/model/best")
     
     # Also save merged model for inference
     model_merged = deepcopy(model)
     model_merged.encoder = model_merged.encoder.merge_and_unload()
     model_merged.decoder = model_merged.decoder.merge_and_unload()
     model_merged.save_pretrained("/workspace/pandith/model/best_merged")

  0%|          | 0/464 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [150]:
import pandas as pd

def import_dataset(path):
  df = pd.read_csv(path, sep='\t', usecols=[0,1])
  #df = pd.read_csv(path, sep='\t', usecols=[0,1])
  df.rename(columns={'file': 'file_name', 'labels': 'text'}, inplace=True)
  #df = df.iloc[1:]
  return(df)

In [176]:
train_df = import_dataset('/workspace/pandith/real/train.tsv')

#train_df = train_df.truncate(after=000)
train_df.head()

,file_name,text
0,29-3.jpg,ප්‍රතිවේධය
1,29-8.jpg,හන්තානයේ
2,4~69.jpg,මියයාම
3,23-9.jpg,කෘශදේහිකය
4,7~65.jpg,ගයන්තිකාගෙන්


In [177]:
test_df = import_dataset('/workspace/pandith/real/test.tsv')

#train_df = train_df.truncate(after=5000)
test_df.head()

,file_name,text
0,10~87.jpg,ෂ්වච්ඡතාව
1,4~88.jpg,ඇවිශ්ශීම
2,1~84.jpg,සණ්දර්භය
3,09-7.jpg,උසුරුවීම
4,25-4.jpg,කන්කැක්කුම


In [258]:
import os
import pandas as pd
from PIL import Image, ImageEnhance, ImageFilter, ImageOps
import numpy as np
import random
import skimage.filters
from skimage.util import random_noise
from tqdm import tqdm
import colorsys
import cv2
from skimage.filters import threshold_sauvola
from skimage import img_as_ubyte

def to_apply():
    return random.choice([True, False])


def noisy(noise_typ, image):
    # Convert PIL image to numpy array
    img_array = np.array(image)

    if noise_typ == "gauss":
        noisy_array = random_noise(img_array, mode="gaussian")
    elif noise_typ == "s&p":
        noisy_array = random_noise(img_array, mode="s&p")
    elif noise_typ == "poisson":
        noisy_array = random_noise(img_array, mode="poisson")

    # random_noise returns float values in [0, 1], convert to uint8 for PIL
    noisy_array = (noisy_array * 255).astype(np.uint8)

    # Convert back to PIL image and return
    return Image.fromarray(noisy_array)


# Function to apply random enhancements to an image
def apply_random_enhancements(image):

    # Randomly adjust rotation make sure to adjust the image size
    rotation = random.randint(-10, 10)
    image = image.rotate(rotation, expand=True, fillcolor=(255, 255, 255))

    if to_apply():
        # Randomly adjust color balance
        color_factor = random.uniform(1, 5)
        enhancer = ImageEnhance.Color(image)
        image = enhancer.enhance(color_factor)

    if to_apply():
        # Apply a randomly colored filter using HSL
        h = random.uniform(0, 1)  # Hue: 0 to 1
        s = random.uniform(0.43, 1)  # Saturation: 50% to 100%
        l = random.uniform(0.2, 0.6)  # Lightness: 50% to 80%

        # Convert HSL to RGB (values will be in range 0-1)
        r, g, b = colorsys.hls_to_rgb(h, l, s)

        # Scale RGB values to 0-255
        r, g, b = int(r * 255), int(g * 255), int(b * 255)

        # Create a color layer with the HSL-based color
        color_layer = Image.new("RGB", image.size, (r, g, b))

        # Random transparency for the color filter
        alpha = random.uniform(0.1, 0.6)  # Subtle effect with alpha between 10% and 50%
        image = Image.blend(image, color_layer, alpha)

    if to_apply():
        # Randomly adjust brightness
        brightness_factor = random.uniform(0.5, 1.5)
        enhancer = ImageEnhance.Brightness(image)
        image = enhancer.enhance(brightness_factor)

    if to_apply():
        # randomly adjust sharpness
        sharpness_factor = random.uniform(0.5, 1.5)
        enhancer = ImageEnhance.Sharpness(image)
        image = enhancer.enhance(sharpness_factor)

    if to_apply():
        # Randomly adjust contrast
        contrast_factor = random.uniform(0.5, 1.5)
        enhancer = ImageEnhance.Contrast(image)
        image = enhancer.enhance(contrast_factor)

    if to_apply():
        # randomly add blur
        blur_factor = random.uniform(0, 2)
        image = image.filter(ImageFilter.GaussianBlur(blur_factor))

    if to_apply():
        noise = random.choice(["gauss", "poisson", "s&p"])
        image = noisy(noise, image)

    return image


def binarize_image(image):

    image = image.copy()  # Create a copy

    # Convert to grayscale
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Compute Sauvola threshold
    window_size = 25  # Can be tuned depending on image size
    sauvola_thresh = threshold_sauvola(gray, window_size=window_size)

    # Apply the threshold
    binary = (gray > sauvola_thresh).astype(np.uint8) * 255  # Binary image: 0 or 255

    binary_rgb = cv2.cvtColor(binary, cv2.COLOR_GRAY2RGB,)
    return binary_rgb

def preprocess(img):
    img = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
    img = binarize_image(img)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = Image.fromarray(img)
    return img

In [300]:
import torch
from torch.utils.data import Dataset
from PIL import Image

class IAMDataset(Dataset):
    def __init__(self, root_dir, df, processor, eval=False, max_target_length=128,):
        self.root_dir = root_dir
        self.df = df
        self.processor = processor
        self.max_target_length = max_target_length
        self.eval = eval

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # get file name + text
        file_name = self.df['file_name'][idx]
        if ".jpg" not in file_name:
          file_name = (file_name.split(".")[0]) + ".jpg"
        text = self.df['text'][idx]

        if text != "":
            # prepare image (i.e. resize + normalize)
            image = Image.open(self.root_dir + file_name).convert("RGB")
            image = preprocess(image)
            if self.eval:
                image = apply_random_enhancements(image)
            pixel_values = self.processor(image, return_tensors="pt").pixel_values
            # add labels (input_ids) by encoding the text
            labels = self.processor.tokenizer(text,
                                              padding="max_length",
                                              max_length=self.max_target_length).input_ids
            # important: make sure that PAD tokens are ignored by the loss function
            labels = [label if label != self.processor.tokenizer.pad_token_id else -100 for label in labels]
    
            encoding = {"pixel_values": pixel_values.squeeze(), "labels": torch.tensor(labels)}
            return encoding

Let's initialize the training and evaluation datasets:

In [301]:

train_dataset = IAMDataset(root_dir='/workspace/pandith/real/train/',
                           df=train_df,
                           processor=processor, eval=True)

eval_dataset = IAMDataset(root_dir='/workspace/pandith/real/test/',
                           df=test_df,
                           processor=processor)

In [302]:
print("Number of training examples:", len(train_dataset))
print("Number of validation examples:", len(eval_dataset))

Number of training examples: 532
Number of validation examples: 133


Let's verify an example from the training dataset:

In [303]:
encoding = train_dataset[1]
for k,v in encoding.items():
  print(k, v.shape)

pixel_values torch.Size([3, 128, 128])
labels torch.Size([128])


We can also check the original image and decode the labels:

In [304]:
labels = encoding['labels']
labels[labels == -100] = processor.tokenizer.pad_token_id
label_str = processor.decode(labels, skip_special_tokens=True)
print(label_str)

හන්තානයේ


Let's create corresponding dataloaders:

In [305]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(train_dataset, batch_size=8, shuffle=True)
eval_dataloader = DataLoader(eval_dataset, batch_size=8)

## Train a model

Here, we initialize the TrOCR model from its pretrained weights. Note that the weights of the language modeling head are already initialized from pre-training, as the model was already trained to generate text during its pre-training stage. Refer to the paper for details.

In [319]:
from transformers import VisionEncoderDecoderModel
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.compile()

Importantly, we need to set a couple of attributes, namely:
* the attributes required for creating the `decoder_input_ids` from the `labels` (the model will automatically create the `decoder_input_ids` by shifting the `labels` one position to the right and prepending the `decoder_start_token_id`, as well as replacing ids which are -100 by the pad_token_id)
* the vocabulary size of the model (for the language modeling head on top of the decoder)
* beam-search related parameters which are used when generating text.

We will evaluate the model on the Character Error Rate (CER), which is available in HuggingFace Datasets (see [here](https://huggingface.co/metrics/cer)).

In [307]:
import evaluate

cer_metric = evaluate.load("cer")

In [308]:
def compute_cer(pred_ids, label_ids):
    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)

    cer = cer_metric.compute(predictions=pred_str, references=label_str)

    return cer

In [309]:
import torch.optim.lr_scheduler as lr_scheduler

def get_current_lr(optimizer):
    """Get the current learning rate from the optimizer."""
    for param_group in optimizer.param_groups:
        return param_group['lr']

In [310]:
from copy import deepcopy

In [321]:
from torch.optim import AdamW
from tqdm.notebook import tqdm

optimizer = AdamW(model.parameters(), lr=5e-5)

# Create adaptive learning rate scheduler that reduces LR when validation metric stops improving
# Note: removed 'verbose' parameter which isn't supported in this PyTorch version
scheduler = lr_scheduler.ReduceLROnPlateau(
    optimizer, 
    mode='min',              # Lower CER is better
    factor=0.5,              # Reduce LR by half when plateauing
    patience=2,              # Wait for 2 epochs before reducing LR
    threshold=0.01,          # Significant change threshold
    threshold_mode='rel',    # Relative threshold
    min_lr=1e-6              # Don't go below this LR
)

best_cer = float('inf')
for epoch in range(10):  # loop over the dataset multiple times
   epoch = epoch + 20
   model.to(device)
   model.train()
   train_loss = 0.0
   for batch in tqdm(train_dataloader):
      for k,v in batch.items():
        batch[k] = v.to(device)
      outputs = model(**batch,interpolate_pos_encoding=True)
      loss = outputs.loss
      loss.backward()
      optimizer.step()
      optimizer.zero_grad()
      train_loss += loss.item()

   avg_train_loss = train_loss/len(train_dataloader)
   print(f"Loss after epoch {epoch}:", avg_train_loss)
   print(f"Current learning rate: {get_current_lr(optimizer)}")
   
   # Save checkpoint after each epoch for resuming training later
   save_peft_model(model, "/workspace/pandith/model/checkpoint", epoch)
   
   # Create a merged model copy for evaluation
   model2 = deepcopy(model)
   model2.encoder = model2.encoder.merge_and_unload()
   model2.decoder = model2.decoder.merge_and_unload()
   model2.to(device)

   # evaluate
   model.eval()
   valid_cer = 0.0
   with torch.no_grad():
     for batch in tqdm(eval_dataloader):
       for k,v in batch.items():
         batch[k] = v.to(device)
       outputs = model2.generate(batch["pixel_values"],interpolate_pos_encoding=True)
       cer = compute_cer(pred_ids=outputs, label_ids=batch["labels"])
       valid_cer += cer 
   
   avg_valid_cer = valid_cer / len(eval_dataloader)
   print("Validation CER:", avg_valid_cer)

   del(model2)
   torch.cuda.empty_cache()
   
   # Store current LR to detect changes
   old_lr = get_current_lr(optimizer)
   
   # Step the scheduler with validation metric
   scheduler.step(avg_valid_cer)
   
   # Check if LR changed and print message
   new_lr = get_current_lr(optimizer)
   if new_lr != old_lr:
       print(f"Learning rate reduced from {old_lr} to {new_lr}")
   
   # Save model if it's the best so far - save both merged model and PEFT adapters
   if avg_valid_cer < best_cer:
     best_cer = avg_valid_cer
     print(f"New best CER: {best_cer}! Saving model...")
     
     # Save PEFT adapters for continued training
     save_peft_model(model, "/workspace/pandith/model/best")
     
     # Also save merged model for inference
     model_merged = deepcopy(model)
     model_merged.encoder = model_merged.encoder.merge_and_unload()
     model_merged.decoder = model_merged.decoder.merge_and_unload()
     model_merged.save_pretrained("/workspace/pandith/model/best_merged")

  0%|          | 0/67 [00:00<?, ?it/s]

Loss after epoch 20: 3.817503879319376
Current learning rate: 5e-05
PEFT model adapters saved to /workspace/pandith/model/checkpoint/epoch_20


  0%|          | 0/17 [00:00<?, ?it/s]

Validation CER: 0.7101055253439792
New best CER: 0.7101055253439792! Saving model...
PEFT model adapters saved to /workspace/pandith/model/best


  0%|          | 0/67 [00:00<?, ?it/s]

...

Validation CER: 0.6797724942261992
Learning rate reduced from 2.5e-05 to 1.25e-05


  0%|          | 0/67 [00:00<?, ?it/s]

Loss after epoch 29: 3.4133545391595184
Current learning rate: 1.25e-05
PEFT model adapters saved to /workspace/pandith/model/checkpoint/epoch_29


  0%|          | 0/17 [00:00<?, ?it/s]

Validation CER: 0.6628008865924527


In [63]:
import torch
torch.cuda.empty_cache()

In [314]:
model.to(device)
model.eval()

print("Ground Truth vs Predicted Text:")
with torch.no_grad():
    for batch in tqdm(train_dataloader):
        # Move inputs to device
        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"]
        
        # Generate predictions
        outputs = model2.generate(pixel_values, interpolate_pos_encoding=True)
        
        # Decode predictions
        pred_texts = processor.batch_decode(outputs, skip_special_tokens=True)
        
        # Fix ground truth labels - replace -100 with pad_token_id before decoding
        labels_for_decoding = labels.clone()
        labels_for_decoding[labels_for_decoding == -100] = processor.tokenizer.pad_token_id
        ground_truth_texts = processor.batch_decode(labels_for_decoding, skip_special_tokens=True)
        
        # Print ground truth and predictions
        for gt, pred in zip(ground_truth_texts, pred_texts):
            print(f"Ground Truth: {gt}")
            print(f"Predicted: {pred}")
            print("-" * 50)

Ground Truth vs Predicted Text:


  0%|          | 0/67 [00:00<?, ?it/s]

Ground Truth: නික්මේ
Predicted: නික්ම
--------------------------------------------------
Ground Truth: අශීර්වාදළත්
Predicted: චාගෙන්ගේගේ
--------------------------------------------------
Ground Truth: මෙහෙශිය
Predicted: මෙෞශි
--------------------------------------------------
Ground Truth: ප්රමාදකාරී
Predicted: සුවතාා
--------------------------------------------------
Ground Truth: ඉණ්ද්රජාලකථණය
Predicted: ඉජණ්ජණණය
--------------------------------------------------
Ground Truth: ජාලයක
Predicted: ජාලයක
--------------------------------------------------
Ground Truth: රිදවා
Predicted: රිද
--------------------------------------------------
Ground Truth: සිසුවියන්ද
Predicted: සිසුවියදද
--------------------------------------------------
Ground Truth: ශිවුරළු
Predicted: ශීුවේ
--------------------------------------------------
Ground Truth: සුවිසුද්ධ
Predicted: සුවටට
--------------------------------------------------
Ground Truth: සූරතිස්ස
Predicted: අරයය
-------------------------------------

KeyboardInterrupt: 

In [ ]:
# Example: Load a checkpoint for inference
checkpoint_path = "/workspace/pandith/model/best"  # or use a specific epoch: "/workspace/pandith/model/checkpoint/epoch_3"
base_model_path = "ds1219/Pandithaya-Printed-Words-128"

# For inference with a merged model (faster)
if os.path.exists("/workspace/pandith/model/best_merged"):
    inference_model = VisionEncoderDecoderModel.from_pretrained("/workspace/pandith/model/best_merged")
    print("Loaded merged model for inference")
else:
    # For inference with adapters (allows further training)
    inference_model = load_peft_model(base_model_path, checkpoint_path)
    print("Loaded model with adapters")

inference_model.to(device)
inference_model.eval()

# Example inference
with torch.no_grad():
    batch = next(iter(eval_dataloader))
    # Move input to device
    pixel_values = batch["pixel_values"].to(device)
    # Generate text
    outputs = inference_model.generate(pixel_values, interpolate_pos_encoding=True)
    # Decode predictions
    pred_str = processor.batch_decode(outputs, skip_special_tokens=True)
    print("Sample predictions:")
    for i, text in enumerate(pred_str[:3]):  # Show first 3 examples
        print(f"{i+1}: {text}")